# Sakana Fugu — multi-agent orchestration behind one model API

[Sakana Fugu](https://sakana.ai/) (announced June 2026) packages a *full
multi-agent orchestration system as a single model API*. You call one endpoint;
internally a **learned orchestrator** decides whether to answer directly or to
assemble a team of expert models from a swappable, multi-vendor pool, then
synthesizes their work into one answer. Its headline claim is **resilience to
single-vendor disruption**: if a provider becomes unavailable (export controls,
outage, opt-out), Fugu dynamically *routes around* it to another model in the pool.

This notebook rebuilds that load-bearing architecture minimally with the Vidbyte
SDK. The whole orchestrator is one small Python class over a few `BaseAgent`s.

```
                       +----------------------- Fugu.run(prompt) -----------------------+
  prompt -> ORCHESTRATOR|  "direct"   -> one available model ----------------------> answer
  (one API) (output_schema|  "delegate" -> per subtask: resolve(assigned_to)            |
            = Plan)       |               [unavailable? ROUTE AROUND] -> pool model     |
                       |               (assigned_to=="fugu" & depth<MAX -> recurse)     |
                       |               gather -> SYNTHESIZER -> (Ultra? -> VERIFIER) -> answer
                       +----------------------------------------------------------------+
   pool = [anthropic:reasoning, openai:coding, gemini:long_context, deepseek:fast]  (swappable)
   available = (key not opted out) AND (provider API key present)
```

**SDK primitives this notebook showcases:**
- `BaseAgent(provider=, model_name=)` across **multiple vendors** as a swappable expert pool.
- `output_schema` + Pydantic — the orchestrator emits a *typed routing plan* (`reply.metadata["structured"]`).
- A plain-Python **class orchestrator** (like `self-refine`'s loop) — typed objects and conditional/recursive control flow cross stage boundaries, which string-only pipelines can't carry.
- Budget + resilience **middleware** (`TokenBudgetMiddleware`, `CostBudgetMiddleware`, `RuntimeLimitMiddleware`, `ModelRetryMiddleware`).
- `@tool` keyless Wikipedia tools so delegated experts produce grounded output.

> **Before running:** copy `.env.example` to `.env` at the repo root and add your
> provider key. **The route-around feature is demonstrable with only
> `OPENAI_API_KEY`** — the other vendors are then "unavailable" and Fugu re-routes
> their subtasks to OpenAI.

## 1. Environment & constants

| Constant | What it controls |
|---|---|
| `PROVIDER` / `MODEL` | The vendor/model for the orchestrator, synthesizer, verifier, and the OpenAI pool member. |
| `TOKEN_BUDGET` | Hard token cap per agent (one runaway agent can't drain the run). |
| `MAX_COST_PER_AGENT` / `COST_PER_MTOK` | Dollar cap per agent; `CostBudgetMiddleware` turns tokens into dollars via your model's blended $/1M-token rate. |
| `MAX_MODEL_CALLS` | Wall-clock-independent cap on model calls per agent. |
| `MAX_TOOL_ROUNDS` | How many search -> read cycles each pool expert may run. |

In [ ]:
%pip install -q vidbyte-sdk python-dotenv requests pydantic

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()
load_dotenv("../../.env")

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL    = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")

TOKEN_BUDGET       = 20_000   # per-agent token cap (middleware)
MAX_COST_PER_AGENT = 0.30     # per-agent dollar cap (middleware)
COST_PER_MTOK      = 4.0      # rough blended $ / 1M tokens for MODEL; set for your model
MAX_MODEL_CALLS    = 12       # per-agent model-call cap (middleware)
MAX_TOOL_ROUNDS    = 5        # search -> read cycles each pool expert may run

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + overrides) in .env first."

## 2. System prompts

Three prompts (Identity -> Goal -> Checklist, the SDK house pattern):

- **Orchestrator** — *the "Fugu model."* It does not answer the user; it returns a
  typed routing plan. Its checklist encodes the routing policy and the exact set
  of specialty keys it may assign to. **The keys here must match the pool keys in
  Section 6** (`reasoning`, `coding`, `long_context`, `fast`, and `fugu` to recurse).
- **Synthesizer** — merges the experts' labeled outputs into one authoritative
  answer and reconciles conflicts. It never sees the orchestration machinery.
- **Verifier** — *Ultra only.* Checks the synthesized answer against the request,
  fixes errors, and returns the final answer.

In [ ]:
ORCHESTRATOR_PROMPT = """# Identity
You are Fugu, an orchestration model. You do NOT answer the user's request
yourself. You decide HOW the request should be handled and return a typed plan.

# Goal
Read the request and choose a route. Use "direct" when a single capable model
can answer well (simple facts, short rewrites, one-domain questions). Use
"delegate" when the request is hard, multi-step, or spans domains: break it into
3-5 self-contained subtasks and assign each to the specialty best suited to it.

# Available specialty keys (assign each subtask to exactly one)
- reasoning      : deep analysis, math, multi-constraint logic
- coding         : software, code review, technical implementation
- long_context   : reading/synthesizing large or many sources
- fast           : quick, low-stakes drafting and formatting
- fugu           : hand a hard sub-problem back to Fugu itself (recursive)

# Checklist
- Prefer "direct" for anything one good model can finish in a single pass
- When delegating, make subtasks independent and jointly sufficient
- Assign each subtask to the single most appropriate specialty key above
- Keep the plan tight: no padding subtasks, no overlap
- Put a one-sentence justification in 'reasoning'"""

SYNTHESIZER_PROMPT = """# Identity
You are the synthesis stage of a multi-agent system. You receive a user request
and a set of expert outputs, each labeled with the expert that produced it.

# Goal
Combine the expert outputs into one authoritative, well-structured answer to the
original request. Reconcile disagreements explicitly, drop redundancy, and do not
introduce claims that no expert supports.

# Checklist
- Answer the ORIGINAL request, not each fragment in isolation
- Resolve conflicts between experts and say which you trust and why
- Preserve concrete facts/citations the experts grounded
- Be complete but do not pad; no meta-commentary about the experts"""

VERIFIER_PROMPT = """# Identity
You are the verification stage of Fugu Ultra, the final quality gate.

# Goal
Check the draft answer against the original request. Fix factual errors, fill
gaps, and tighten structure. Return ONLY the corrected final answer.

# Checklist
- Verify every requirement in the request is actually met
- Correct anything wrong or unsupported; do not add unrelated content
- Return the final answer text only, with no notes about your edits"""

## 3. Tools

Fugu's real "tools" are the **models in its pool** — orchestration, not tool use,
is the product. But to make delegated subtasks produce *grounded* output, each
pool expert gets two keyless tools over Wikipedia's public API (`search`,
`read_article`) — the same toolbox the other cookbook notebooks use, so this runs
without a search-API key. Swap them for filesystem/code/SQL tools and the
orchestration is unchanged. The orchestrator, synthesizer, and verifier get **no
tools** — they reason over text only.

In [ ]:
import re, requests
from vidbyte import tool

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS  = {"User-Agent": "vidbyte-cookbook-sakana-fugu/0.1"}


@tool
def search(query: str) -> list[dict]:
    """Search Wikipedia for articles matching the query. Returns up to 5 results with
    title and a short snippet. Use read_article to read one in full."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "list": "search", "srsearch": query, "srlimit": 5, "format": "json",
    })
    resp.raise_for_status()
    return [{"title": h["title"], "snippet": re.sub(r"<[^>]+>", "", h["snippet"])}
            for h in resp.json()["query"]["search"]]


@tool
def read_article(title: str) -> str:
    """Read the plain-text extract of one Wikipedia article by its exact title (truncated
    to 6 000 chars). Use the exact title returned by search()."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "prop": "extracts", "explaintext": 1,
        "titles": title, "format": "json", "redirects": 1,
    })
    resp.raise_for_status()
    pages = resp.json()["query"]["pages"]
    extract = next(iter(pages.values())).get("extract", "")
    return extract[:6_000] if extract else f"No article found for '{title}'."


POOL_TOOLS = [search, read_article]

## 4. The orchestration core — `Fugu`

This is the whole product in one class. The cells below are marked
`# [fugu-core]`: they import only the standard library + Pydantic (no `vidbyte`),
so the verification script in `scripts/test_sakana_fugu.py` can exec exactly
these cells and test the routing logic offline with fake agents.

**Typed plan.** `OrchestrationPlan` is the orchestrator's `output_schema`: a
`mode` plus, when delegating, a list of `Subtask`s each carrying an `assigned_to`
specialty key. **Pool.** Each `PoolModel` binds a specialty key to a vendor,
model id, and a runnable agent. **Availability** is the load-bearing idea: a
model is usable only if it isn't opted out *and* its provider's API key is
present — which is exactly what makes **route-around** real.

In [ ]:
# [fugu-core]
import os
from dataclasses import dataclass
from typing import Any
from pydantic import BaseModel, Field

# Which env var proves a vendor is reachable. A model counts as "available" only
# if its provider key is present here -- this is what drives route-around.
PROVIDER_ENV = {
    "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY",
    "gemini": "GEMINI_API_KEY", "xai": "XAI_API_KEY", "deepseek": "DEEPSEEK_API_KEY",
}
MAX_RECURSION_DEPTH = 2   # cap on Fugu calling itself, so recursion always terminates


class Subtask(BaseModel):
    description: str = Field(description="A self-contained unit of work for one expert model.")
    assigned_to: str = Field(description="Pool specialty key for this subtask, e.g. 'reasoning' (or 'fugu' to recurse).")


class OrchestrationPlan(BaseModel):
    mode: str = Field(description='Exactly "direct" (answer with one model) or "delegate" (assemble a team).')
    reasoning: str = Field(description="One sentence explaining the routing choice.")
    subtasks: list[Subtask] = Field(default_factory=list, description="The team's subtasks when delegating; empty when direct.")


@dataclass
class PoolModel:
    """One swappable expert: a specialty key bound to a vendor, model id, and a runnable agent."""
    key: str
    provider: str
    model: str
    agent: Any   # anything exposing .run(str) -> reply with a .content attribute

In [ ]:
# [fugu-core]
class Fugu:
    """A multi-agent orchestration system that behaves like a single model: one run() in, one answer out."""

    def __init__(self, orchestrator, pool, synthesizer, verifier=None, *, ultra=False, exclude=(), verbose=False):
        # Wire the Fugu model, the swappable expert pool, and the synthesis/verification stages.
        self.orchestrator = orchestrator
        self.pool = list(pool)
        self.by_key = {m.key: m for m in self.pool}
        self.synthesizer = synthesizer
        self.verifier = verifier
        self.ultra = ultra
        self.exclude = set(exclude)
        self.verbose = verbose

    def run(self, prompt, _depth=0):
        # The single-model API: orchestrate internally and return exactly one answer.
        plan = self._orchestrate(prompt)
        self._say(f"route={plan.mode} ({plan.reasoning})")
        if plan.mode != "delegate" or not plan.subtasks:
            return self._solve_directly(prompt)
        outputs = self._dispatch(plan.subtasks, _depth)
        answer = self._synthesize(prompt, outputs)
        return self._verify(prompt, answer) if (self.ultra and self.verifier) else answer

    def _orchestrate(self, prompt):
        # Ask the Fugu model how to route; fall back to a direct plan if structured output is missing.
        reply = self.orchestrator.run(prompt)
        payload = reply.metadata.get("structured")
        if payload is None:
            return OrchestrationPlan(mode="direct", reasoning="fallback: orchestrator returned no structured plan")
        return payload if isinstance(payload, OrchestrationPlan) else OrchestrationPlan.model_validate(payload)

    def _available(self, key):
        # A model is usable only if it exists, is not opted out, and its provider key is present.
        if key not in self.by_key or key in self.exclude:
            return False
        return bool(os.getenv(PROVIDER_ENV.get(self.by_key[key].provider, "")))

    def _resolve(self, requested_key, _skip=()):
        # Return the assigned model, or route around to the first available one; raise if the pool is dark.
        if requested_key not in _skip and self._available(requested_key):
            return self.by_key[requested_key]
        for member in self.pool:
            if member.key not in _skip and self._available(member.key):
                if member.key != requested_key:
                    self._say(f"routing around '{requested_key}' -> '{member.key}' (assigned provider unavailable or opted out)")
                return member
        raise RuntimeError("Fugu: no available models in pool (every provider is unavailable or opted out)")

    def _run_subtask(self, task, depth):
        # Execute one subtask on its expert; recurse into Fugu when asked, and route around runtime failures.
        if task.assigned_to == "fugu" and depth < MAX_RECURSION_DEPTH:
            self._say(f"recursing into Fugu (depth {depth + 1})")
            return f"[fugu] {self.run(task.description, _depth=depth + 1)}"
        member = self._resolve(task.assigned_to)
        try:
            return f"[{member.key}] {member.agent.run(task.description).content}"
        except Exception as exc:
            self._say(f"'{member.key}' failed at runtime ({exc}); routing around")
            substitute = self._resolve(member.key, _skip={member.key})
            return f"[{substitute.key}] {substitute.agent.run(task.description).content}"

    def _dispatch(self, subtasks, depth):
        # Run every subtask, labeling each output with the model that produced it.
        return [self._run_subtask(task, depth) for task in subtasks]

    def _synthesize(self, prompt, outputs):
        # Merge the experts' labeled outputs into one authoritative answer.
        joined = "\n\n".join(outputs)
        return self.synthesizer.run(f"Original request:\n{prompt}\n\nExpert outputs:\n{joined}").content

    def _verify(self, prompt, answer):
        # Ultra-only: have a verifier check and finalize the synthesized answer.
        self._say("ultra: running verification pass")
        return self.verifier.run(f"Request:\n{prompt}\n\nDraft answer:\n{answer}").content

    def _solve_directly(self, prompt):
        # Answer with a single available model, reusing the route-around resolver via a never-real key.
        return self._resolve("__any__").agent.run(prompt).content

    def _say(self, msg):
        # Print an orchestration trace line when verbose.
        if self.verbose:
            print(f"  [fugu] {msg}")

## 5. Middleware

Middleware is deterministic runtime policy the model never sees. Fugu fans work
out across many agents, so every agent gets its own fresh guards:

| Middleware | Role in Fugu |
|---|---|
| `TokenBudgetMiddleware` | Hard token cap per agent. |
| `CostBudgetMiddleware` | Dollar cap per agent (`max_spend_usd` + `cost_per_million_tokens`). |
| `RuntimeLimitMiddleware` | Caps model calls + wall-clock so one expert can't hang the run. |
| `ModelRetryMiddleware` | Retries transient provider errors — the SDK-level complement to Fugu's route-around. |

> **More middleware:** the SDK ships `CircuitBreakerMiddleware` (open the circuit
> on a flaky provider — a natural pair with route-around), `LoopDetectionMiddleware`,
> `AuditLogMiddleware`, and compaction middlewares. See
> `vidbyte-sdk/skills/vidbyte-sdk/middleware.md`.

In [ ]:
from vidbyte.middleware import (
    TokenBudgetMiddleware,
    CostBudgetMiddleware,
    RuntimeLimitMiddleware,
    ModelRetryMiddleware,
)


def guards():
    """Return a fresh budget + resilience middleware list (new instances so state never leaks between agents)."""
    return [
        TokenBudgetMiddleware(max_tokens=TOKEN_BUDGET),
        CostBudgetMiddleware(max_spend_usd=MAX_COST_PER_AGENT, cost_per_million_tokens=COST_PER_MTOK),
        RuntimeLimitMiddleware(max_model_calls=MAX_MODEL_CALLS, max_elapsed_seconds=120.0),
        ModelRetryMiddleware(max_retries=2),
    ]

## 6. Constructing the agents and the pool

The orchestrator/synthesizer/verifier run on your default `PROVIDER`/`MODEL`. The
**pool** names real, *different* vendors — that is the whole point: "the world's
best models," entirely swappable by editing the list. Each expert is a tool-using
`BaseAgent` with its own budget guards.

The two tiers are the same class behind the same `run` API: `fugu` synthesizes
once; `fugu_ultra` adds a verifier pass. With only `OPENAI_API_KEY` set, only the
`coding` (OpenAI) member is *available* — so any subtask the orchestrator assigns
to `reasoning`/`long_context`/`fast` gets **routed around** to OpenAI.

In [ ]:
from vidbyte import BaseAgent

orchestrator = BaseAgent(name="fugu-orchestrator", system_prompt=ORCHESTRATOR_PROMPT,
                         provider=PROVIDER, model_name=MODEL,
                         output_schema=OrchestrationPlan, middleware=guards())

synthesizer = BaseAgent(name="fugu-synthesizer", system_prompt=SYNTHESIZER_PROMPT,
                        provider=PROVIDER, model_name=MODEL, middleware=guards())

verifier = BaseAgent(name="fugu-verifier", system_prompt=VERIFIER_PROMPT,
                     provider=PROVIDER, model_name=MODEL, middleware=guards())

EXPERT_PROMPT = ("You are a {specialty} specialist inside a multi-agent system. Solve the assigned "
                 "subtask precisely and completely. Use the search/read_article tools to ground factual "
                 "claims. Return a self-contained result; do not ask follow-up questions.")


def make_expert(key, provider, model, specialty):
    """Construct one swappable pool expert as a tool-using BaseAgent on the given vendor."""
    agent = BaseAgent(name=f"expert-{key}", system_prompt=EXPERT_PROMPT.format(specialty=specialty),
                      provider=provider, model_name=model, tools=POOL_TOOLS,
                      middleware=guards(), max_tool_rounds=MAX_TOOL_ROUNDS)
    return PoolModel(key=key, provider=provider, model=model, agent=agent)


# The swappable, multi-vendor pool. Swap a vendor/model on any line to change the pool.
POOL = [
    make_expert("reasoning",    "anthropic", "claude-sonnet-4-6", "deep reasoning and analysis"),
    make_expert("coding",       "openai",    MODEL,               "software engineering"),
    make_expert("long_context", "gemini",    "gemini-2.5-pro",    "long-context synthesis"),
    make_expert("fast",         "deepseek",  "deepseek-chat",     "fast, low-cost drafting"),
]

fugu       = Fugu(orchestrator, POOL, synthesizer, verbose=True)                         # the fast tier
fugu_ultra = Fugu(orchestrator, POOL, synthesizer, verifier=verifier, ultra=True, verbose=True)  # max quality

## 7. Running Fugu + the route-around demo

Call `fugu.run(prompt)` — one method, one answer. Watch the `[fugu]` trace lines:
they show the routing decision and every route-around. Try these:

```
# Simple -> Fugu answers directly with one model
"What year did the Concorde first fly?"

# Hard / multi-domain -> Fugu delegates a team (and routes around missing vendors)
"Compare the economic and technical reasons the Concorde was retired, and outline
 what a modern supersonic revival would need to solve."

# Opt a model out for compliance: Fugu(orchestrator, POOL, synthesizer, exclude={'coding'})
```

In [ ]:
# 1) Simple ask -> Fugu should route "direct" and consult a single model.
print("=== DIRECT ===")
print(fugu.run("What year did the Concorde first fly?"))

In [ ]:
# 2) Hard, multi-domain ask -> Fugu delegates a team. With only OPENAI_API_KEY set,
#    the anthropic/gemini/deepseek experts are unavailable and Fugu ROUTES AROUND
#    them to the OpenAI member -- watch the trace lines.
print("=== DELEGATE (watch the route-around trace) ===")
answer = fugu.run(
    "Compare the economic and technical reasons the Concorde was retired, and "
    "outline what a modern supersonic passenger revival would need to solve."
)
display(Markdown(answer))

In [ ]:
# 3) Inspect the orchestrator's typed plan directly (the structured-output artifact).
plan = orchestrator.run(
    "Compare the economic and technical reasons the Concorde was retired, and "
    "outline what a modern supersonic passenger revival would need to solve."
).metadata["structured"]
print("mode:", plan.mode, "|", plan.reasoning)
for s in plan.subtasks:
    print(f"  - [{s.assigned_to}] {s.description}")

## 8. Example output

With only `OPENAI_API_KEY` set, a delegate run looks like this — note how every
non-OpenAI expert is transparently routed around to the available OpenAI model,
yet the caller still gets a single coherent answer:

```
=== DELEGATE (watch the route-around trace) ===
  [fugu] route=delegate (multi-domain comparison plus forward-looking synthesis)
  [fugu] routing around 'reasoning' -> 'coding' (assigned provider unavailable or opted out)
  [fugu] routing around 'long_context' -> 'coding' (assigned provider unavailable or opted out)
  [fugu] routing around 'fast' -> 'coding' (assigned provider unavailable or opted out)

# Why the Concorde Was Retired — and What a Revival Must Solve

**Economic:** Concorde was profitable only on a few premium transatlantic routes;
fuel burn per seat-mile was several times that of subsonic jets ...

**Technical:** The sonic boom confined supersonic flight to oceanic routes, and
the ageing airframe faced rising maintenance and certification costs ...

**A modern revival would need to solve:** low-boom airframe shaping for overland
routes, sustainable-fuel economics, and updated supersonic certification ...
```

Set `ANTHROPIC_API_KEY` too and re-run: the `reasoning` subtask now resolves to
the Anthropic member instead of routing around — the pool grew, no code changed.

## 9. What production Fugu adds — and next steps

This skeleton is honest about its limits. Production Fugu adds, beyond it:

- **A *learned* coordinator.** Our orchestrator is a prompted LLM emitting a plan;
  Sakana trains the coordinator (their Trinity / Conductor papers) so routing and
  decomposition improve from data rather than a prompt.
- **A real multi-vendor fleet** with live keys, health checks, and an
  OpenAI-compatible HTTP endpoint (here the "single API" is `Fugu.run`).
- **Verification/voting** richer than one Ultra pass — cross-checking experts and
  weighting by confidence.

**Try it yourself:**

1. **Prove route-around the other way.** Add `ANTHROPIC_API_KEY` to `.env` and
   re-run the delegate cell. Which subtasks stop routing around?

2. **Opt a model out.** Build `Fugu(orchestrator, POOL, synthesizer, exclude={"coding"})`
   and run with only `OPENAI_API_KEY`. What does `Fugu.run` do now, and why is that
   the *correct* behavior (see `_resolve`)?

3. **Turn it into a real swarm.** The SDK ships an `ActorRuntime` (one `BaseAgent`
   that dynamically spawns sub-actors). Where would it replace the manual dispatch,
   and what would you lose (hint: middleware compatibility)?

4. **Add a circuit breaker.** Put `CircuitBreakerMiddleware` on each expert. How
   does it interact with `_run_subtask`'s runtime route-around — which fires first?

5. **Best-of instead of synthesize.** Change `_synthesize` to ask the synthesizer
   to *pick* the single best expert output rather than merge. When is each better?